# Testing Card Condition Grader

Notebook ini membaca satu gambar kartu, mengirimkannya ke model Roboflow melalui `card_condition_grader.py`, dan menampilkan hasil JSON. Tidak membuat gambar anotasi.

## 1. Memuat gambar dan modul model

Ganti `TEST_IMAGE_PATH` dengan lokasi gambar yang ingin diuji.

In [15]:
from pathlib import Path
import importlib
import json
import sys

import cv2
from dotenv import load_dotenv

project_root = Path.cwd()
while project_root != project_root.parent:
    if (project_root / 'backend' / 'models' / 'card_condition_grader.py').exists():
        break
    project_root = project_root.parent
else:
    raise FileNotFoundError('Root proyek tidak ditemukan.')

models_dir = project_root / 'backend' / 'models'
load_dotenv(project_root / '.env')
sys.path.insert(0, str(models_dir))
import card_condition_grader
importlib.reload(card_condition_grader)
detect_card_defects = card_condition_grader.detect_card_defects

TEST_IMAGE_PATH = Path(r'D:\Datasets\tes-pokemon.jpg')
if not TEST_IMAGE_PATH.exists():
    raise FileNotFoundError(f'Gambar tidak ditemukan: {TEST_IMAGE_PATH}')

image_bgr = cv2.imread(str(TEST_IMAGE_PATH))
if image_bgr is None:
    raise ValueError(f'OpenCV tidak dapat membaca gambar: {TEST_IMAGE_PATH}')

print(f'Gambar: {TEST_IMAGE_PATH}')
print(f'Resolusi: {image_bgr.shape[1]} x {image_bgr.shape[0]} piksel')

Gambar: D:\Datasets\tes-pokemon.jpg
Resolusi: 1920 x 1440 piksel


## 2. Mengirim gambar ke Roboflow

Fungsi model mengubah isi gambar menjadi Base64 sebelum mengirimkannya ke endpoint Roboflow.

In [16]:
result = detect_card_defects(str(TEST_IMAGE_PATH))
if result is None:
    raise RuntimeError('Model tidak mengembalikan hasil deteksi.')
print(json.dumps(result, indent=2, ensure_ascii=False))

{
  "inference_id": "ec5d362b-4014-4f88-a445-83ca9bfc9c5d",
  "time": 0.06604335200972855,
  "image": {
    "width": 1920,
    "height": 1440
  },
  "predictions": [
    {
      "x": 949.5,
      "y": 782.0,
      "width": 1427.0,
      "height": 1168.0,
      "confidence": 0.974353551864624,
      "class": "Card",
      "class_id": 0,
      "detection_id": "d8ad868e-3a44-40b5-9c20-bcabe8cd5171"
    }
  ]
}


## 3. Ringkasan prediksi

Cell ini menampilkan jumlah prediksi dan kelas defect yang terdeteksi tanpa menyimpan gambar baru.

In [17]:
predictions = result.get('predictions', [])
print(f'Jumlah prediksi: {len(predictions)}')
for index, prediction in enumerate(predictions, start=1):
    label = prediction.get('class', 'unknown')
    confidence = prediction.get('confidence', 0)
    print(f'{index}. {label} (confidence={confidence:.3f})')

Jumlah prediksi: 1
1. Card (confidence=0.974)


## 4. Status kondisi kartu

Cell ini memeriksa secara khusus apakah model mendeteksi `Corner Wear`, `Edge Wear`, atau `Scratch`. Kelas `Card` hanya menunjukkan bahwa objek kartu ditemukan.

In [18]:
defect_classes = {
    'Corner Wear': False,
    'Edge Wear': False,
    'Scratch': False,
}

for prediction in predictions:
    label = prediction.get('class')
    if label in defect_classes:
        defect_classes[label] = True

print('Status kondisi kartu:')
for label, detected in defect_classes.items():
    status = 'TERDETEKSI' if detected else 'TIDAK TERDETEKSI'
    print(f'- {label}: {status}')

has_defect = any(defect_classes.values())
print(f"\nAda defect: {'YA' if has_defect else 'TIDAK TERDETEKSI'}")

Status kondisi kartu:
- Corner Wear: TIDAK TERDETEKSI
- Edge Wear: TIDAK TERDETEKSI
- Scratch: TIDAK TERDETEKSI

Ada defect: TIDAK TERDETEKSI


## 5. Confidence setiap kelas

Roboflow hanya mengembalikan prediksi yang melewati threshold. Karena itu, kelas yang tidak muncul diberi nilai `0.000` sebagai penanda tidak ada prediksi yang dikembalikan; nilai tersebut bukan probabilitas absolut kelas.

In [19]:
classes = ['Card', 'Corner Wear', 'Edge Wear', 'Scratch']
confidence_by_class = {label: 0.0 for label in classes}

for prediction in predictions:
    label = prediction.get('class')
    confidence = float(prediction.get('confidence', 0.0))
    if label in confidence_by_class:
        confidence_by_class[label] = max(confidence_by_class[label], confidence)

print('Confidence maksimum per kelas:')
for label in classes:
    confidence = confidence_by_class[label]
    detected = confidence > 0
    status = 'ada prediksi' if detected else 'tidak ada prediksi dikembalikan'
    print(f'- {label}: {confidence:.3f} ({confidence:.1%}) - {status}')

Confidence maksimum per kelas:
- Card: 0.974 (97.4%) - ada prediksi
- Corner Wear: 0.000 (0.0%) - tidak ada prediksi dikembalikan
- Edge Wear: 0.000 (0.0%) - tidak ada prediksi dikembalikan
- Scratch: 0.000 (0.0%) - tidak ada prediksi dikembalikan
